<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/Optuna/z347_Optuna_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optuna + LightGBM — búsqueda de hiperparámetros

Recibe el dataset de Feature Engineering ya preprocesado y:
1. Detecta y filtra variables con posible data leakage
2. Corre Optuna para optimizar LightGBM
3. Visualiza la evolución de trials y parámetros
4. Entrena el modelo final con los mejores hiperparámetros

**Apartado al final**: warm-starting para clusters — cómo reutilizar lo aprendido en la búsqueda global cuando arrancás una búsqueda por cluster.

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
!pip install uv -q
!uv pip install -q lightgbm optuna optuna-integration[lightgbm] kaggle plotly

In [ ]:
import os, warnings
import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import scipy.stats as stats

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── PARAM ────────────────────────────────────────────────────────────────────
PARAM = {
    # Ruta al dataset de FE preprocesado (recibido del grupo de FE)
    'dataset_path':      '/content/buckets/b1/exp/fe/dataset_fe.csv',

    # Columna target
    'target':            'clase',

    # Columnas que NO son features (id, periodo, target)
    'cols_excluir':      ['product_id', 'periodo', 'clase'],

    # Umbral de correlación con el target para alertar data leakage
    'umbral_leakage':    0.98,

    # Umbral de correlación entre features (multicolinealidad extrema)
    'umbral_multicol':   0.995,

    # Optuna
    'n_trials':          80,
    'n_folds':           5,
    'semilla':           102191,
    'n_jobs_lgbm':       -1,

    # Kaggle
    'competencia':       'labo-iii-2026-rosario',
    'drive_path':        '/content/buckets/b1/exp/optuna',
}

os.makedirs(PARAM['drive_path'], exist_ok=True)
print('OK')

# 1. Carga del dataset de Feature Engineering

In [ ]:
df = pl.read_csv(PARAM['dataset_path'])
print(f'Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Columnas: {df.columns[:10]} ... ({df.shape[1]} total)')
print()
display(df.head(3))

In [ ]:
# separar features y target
cols_features = [c for c in df.columns if c not in PARAM['cols_excluir']]
print(f'Features disponibles: {len(cols_features)}')

# pasar a pandas para sklearn/lgbm
X_all = df.select(cols_features).to_pandas()
y_all = df[PARAM['target']].to_pandas()

print(f'X shape: {X_all.shape}')
print(f'y — media: {y_all.mean():.4f}  std: {y_all.std():.4f}  min: {y_all.min():.4f}  max: {y_all.max():.4f}')

# 2. Detección de data leakage

Una feature tiene data leakage si:
- **Correlación con el target > umbral**: la feature "conoce" el futuro directamente
- **Nombre sospechoso**: contiene `t-1`, `t-2`, `lag_-`, `_futuro`, etc.
- **Multicolinealidad extrema**: dos features casi idénticas (una puede ser derivada de la otra con fuga)

In [ ]:
print('── Análisis de data leakage ──')
print()

leakage_flags = {}

# 1. Correlación con el target
corr_target = X_all.corrwith(y_all).abs().sort_values(ascending=False)
sospechosas_corr = corr_target[corr_target > PARAM['umbral_leakage']].index.tolist()

print(f'1. Correlación con target > {PARAM["umbral_leakage"]}:')
if sospechosas_corr:
    for col in sospechosas_corr:
        print(f'   ⚠  {col:40s}  r={corr_target[col]:.4f}')
else:
    print('   Ninguna — OK')

# 2. Nombres sospechosos (lags negativos = datos del futuro)
patrones_leakage = ['_-', 'lag_-', 'futuro', 't-1', 't-2', 't+', 'forward']
sospechosas_nombre = [c for c in cols_features
                      if any(p in c.lower() for p in patrones_leakage)]
print(f'\n2. Nombres sospechosos (lags negativos / forward):')
if sospechosas_nombre:
    for col in sospechosas_nombre:
        print(f'   ⚠  {col}')
else:
    print('   Ninguna — OK')

# 3. Top 20 correlaciones con target (para inspección manual)
print(f'\n3. Top 20 features por correlación con target:')
print(corr_target.head(20).to_frame('|corr|').to_string())

# Unión de todas las sospechosas
cols_leakage = list(set(sospechosas_corr + sospechosas_nombre))
print(f'\nTotal features con flag de leakage: {len(cols_leakage)}')
if cols_leakage:
    print('  → Serán excluidas del entrenamiento. Revisar manualmente antes de confirmar.')
    print('  ', cols_leakage)

In [ ]:
# Multicolinealidad extrema
print('── Multicolinealidad extrema ──')
corr_matrix = X_all.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
pares_multicol = [(col, row, upper.loc[row, col])
                  for col in upper.columns
                  for row in upper.index
                  if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > PARAM['umbral_multicol']]

if pares_multicol:
    print(f'Pares con correlación > {PARAM["umbral_multicol"]}:')
    for c1, c2, r in sorted(pares_multicol, key=lambda x: -x[2]):
        print(f'  {c1:35s} ↔ {c2:35s}  r={r:.4f}')
else:
    print('Ningún par con multicolinealidad extrema — OK')

In [ ]:
# ── Decisión final: qué columnas usar ──
# Podés agregar o quitar columnas manualmente acá
COLS_EXCLUIR_MANUAL = []  # ← agregar nombres si querés forzar exclusión

cols_leakage_final = list(set(cols_leakage + COLS_EXCLUIR_MANUAL))
cols_train = [c for c in cols_features if c not in cols_leakage_final]

X = X_all[cols_train]
y = y_all

print(f'Features originales:   {len(cols_features)}')
print(f'Excluidas por leakage: {len(cols_leakage_final)}')
print(f'Features para train:   {len(cols_train)}')

# 3. Optuna — búsqueda de hiperparámetros LightGBM

In [ ]:
def objective(trial):
    params = {
        'objective':        'regression',
        'metric':           'rmse',
        'verbosity':        -1,
        'boosting_type':    'gbdt',
        'n_jobs':           PARAM['n_jobs_lgbm'],
        'random_state':     PARAM['semilla'],

        # hiperparámetros a optimizar
        'n_estimators':     trial.suggest_int('n_estimators', 100, 2000),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 16, 256),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 100),
        'subsample':        trial.suggest_float('subsample', 0.4, 1.0),
        'subsample_freq':   1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain':   trial.suggest_float('min_split_gain', 0.0, 1.0),
    }

    kf = KFold(n_splits=PARAM['n_folds'], shuffle=True, random_state=PARAM['semilla'])
    rmses = []

    for fold, (idx_tr, idx_val) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[idx_tr], X.iloc[idx_val]
        y_tr, y_val = y.iloc[idx_tr], y.iloc[idx_val]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        pred = model.predict(X_val)
        rmse = mean_squared_error(y_val, pred, squared=False)
        rmses.append(rmse)

        # pruning: si el fold va mal, cortar antes
        trial.report(np.mean(rmses), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(rmses))


print('Función objetivo definida.')

In [ ]:
# Crear y correr el estudio
sampler = optuna.samplers.TPESampler(seed=PARAM['semilla'])
pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)

study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    pruner=pruner,
    study_name='lgbm_global',
)

print(f'Corriendo {PARAM["n_trials"]} trials...')
study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=True)

print(f'\nMejor RMSE: {study.best_value:.4f}')
print(f'Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k:20s}: {v}')

# 4. Visualización — evolución de trials y parámetros

In [ ]:
# ── Evolución del RMSE por trial ──
trials_df = study.trials_dataframe()
trials_ok  = trials_df[trials_df['state'] == 'COMPLETE'].copy()
trials_ok['best_so_far'] = trials_ok['value'].cummin()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['RMSE por trial', 'Mejor RMSE acumulado'])

fig.add_trace(go.Scatter(
    x=trials_ok['number'], y=trials_ok['value'],
    mode='markers', marker=dict(size=4, color='steelblue', opacity=0.6),
    name='RMSE trial',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=trials_ok['number'], y=trials_ok['best_so_far'],
    mode='lines', line=dict(color='tomato', width=2),
    name='mejor hasta ahora',
), row=1, col=1)

# distribución de RMSE
fig.add_trace(go.Histogram(
    x=trials_ok['value'], nbinsx=30,
    marker_color='steelblue', opacity=0.7,
    name='distribución RMSE',
), row=1, col=2)

fig.add_vline(x=study.best_value, line_dash='dash', line_color='tomato',
              annotation_text=f'mejor={study.best_value:.4f}', row=1, col=2)

fig.update_layout(title='Evolución de trials Optuna', height=400, showlegend=True)
fig.show()

In [ ]:
# ── Importancia de hiperparámetros ──
importances = optuna.importance.get_param_importances(study)

fig = go.Figure(go.Bar(
    x=list(importances.values()),
    y=list(importances.keys()),
    orientation='h',
    marker_color='steelblue',
))
fig.update_layout(
    title='Importancia de hiperparámetros (fANOVA)',
    xaxis_title='importancia relativa',
    height=400,
    yaxis=dict(autorange='reversed'),
)
fig.show()
print('Hiperparámetros con mayor importancia = los que más impactan en el RMSE final.')
print('Si un parámetro tiene importancia baja, podés fijarlo y ahorrar trials.')

In [ ]:
# ── Evolución de cada hiperparámetro por trial ──
param_cols = [c for c in trials_ok.columns if c.startswith('params_')]
n_params   = len(param_cols)
ncols = 3
nrows = (n_params + ncols - 1) // ncols

fig = make_subplots(rows=nrows, cols=ncols,
    subplot_titles=[c.replace('params_', '') for c in param_cols])

rmse_norm = (trials_ok['value'] - trials_ok['value'].min()) / (trials_ok['value'].max() - trials_ok['value'].min())
colors = px.colors.sample_colorscale('RdYlGn_r', rmse_norm.tolist())

for i, col in enumerate(param_cols):
    r, c = divmod(i, ncols)
    fig.add_trace(go.Scatter(
        x=trials_ok['number'],
        y=trials_ok[col],
        mode='markers',
        marker=dict(size=5, color=trials_ok['value'],
                    colorscale='RdYlGn_r', showscale=(i==0),
                    colorbar=dict(title='RMSE') if i==0 else None),
        showlegend=False,
    ), row=r+1, col=c+1)

fig.update_layout(
    title='Evolución de hiperparámetros por trial (color = RMSE: verde=bueno, rojo=malo)',
    height=200 * nrows,
)
fig.show()

In [ ]:
# ── Parallel coordinates — los mejores 30 trials ──
top30 = trials_ok.nsmallest(30, 'value')

dims = [dict(label='RMSE', values=top30['value'])]
for col in param_cols:
    vals = top30[col].dropna()
    if vals.nunique() > 1:
        dims.append(dict(label=col.replace('params_', ''), values=top30[col]))

fig = go.Figure(go.Parcoords(
    line=dict(color=top30['value'], colorscale='RdYlGn_r',
              showscale=True, colorbar=dict(title='RMSE')),
    dimensions=dims,
))
fig.update_layout(
    title='Parallel coordinates — top 30 trials (arrastrá los ejes para filtrar)',
    height=450,
)
fig.show()

# 5. Modelo final — entrenamiento con mejores hiperparámetros

In [ ]:
best_params = {
    **study.best_params,
    'objective':     'regression',
    'metric':        'rmse',
    'verbosity':     -1,
    'boosting_type': 'gbdt',
    'n_jobs':        PARAM['n_jobs_lgbm'],
    'random_state':  PARAM['semilla'],
    'subsample_freq': 1,
}

# CV final con mejores parámetros
kf = KFold(n_splits=PARAM['n_folds'], shuffle=True, random_state=PARAM['semilla'])
rmses_final = []
modelos_fold = []

for fold, (idx_tr, idx_val) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[idx_tr], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_tr], y.iloc[idx_val]

    m = lgb.LGBMRegressor(**best_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(-1)])
    pred  = m.predict(X_val)
    rmse  = mean_squared_error(y_val, pred, squared=False)
    rmses_final.append(rmse)
    modelos_fold.append(m)
    print(f'  Fold {fold+1}: RMSE = {rmse:.4f}')

print(f'\nRMSE final CV: {np.mean(rmses_final):.4f} ± {np.std(rmses_final):.4f}')

In [ ]:
# ── Feature importance promedio entre folds ──
importances_lgbm = np.mean(
    [m.feature_importances_ for m in modelos_fold], axis=0
)
fi_df = pd.DataFrame({'feature': cols_train, 'importance': importances_lgbm})
fi_df = fi_df.sort_values('importance', ascending=False).head(30)

fig = go.Figure(go.Bar(
    x=fi_df['importance'],
    y=fi_df['feature'],
    orientation='h',
    marker_color='steelblue',
))
fig.update_layout(
    title='Feature importance LightGBM — top 30 (promedio entre folds)',
    xaxis_title='importance (gain)',
    height=max(400, len(fi_df) * 18),
    yaxis=dict(autorange='reversed'),
)
fig.show()

In [ ]:
# ── Predicción y residuos ──
# usar el modelo del fold 1 para visualizar
m_vis = modelos_fold[0]
idx_tr0, idx_val0 = list(kf.split(X))[0]
pred_val = m_vis.predict(X.iloc[idx_val0])
y_val0   = y.iloc[idx_val0].values
residuos = pred_val - y_val0

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Predicho vs real (fold 1)', 'Distribución de residuos'])

fig.add_trace(go.Scatter(
    x=y_val0, y=pred_val, mode='markers',
    marker=dict(size=3, opacity=0.4, color='steelblue'),
    name='pred vs real',
), row=1, col=1)
lim = max(y_val0.max(), pred_val.max())
fig.add_trace(go.Scatter(x=[0,lim], y=[0,lim], mode='lines',
    line=dict(dash='dash', color='gray'), name='perfecto'), row=1, col=1)

fig.add_trace(go.Histogram(
    x=residuos, nbinsx=50,
    marker_color='steelblue', opacity=0.7, name='residuos',
), row=1, col=2)
fig.add_vline(x=0, line_dash='dash', line_color='tomato', row=1, col=2)

fig.update_layout(title='Calidad del modelo — fold 1', height=400)
fig.show()

# 6. Guardar estudio y mejores parámetros

In [ ]:
import json, shutil

# guardar mejores parámetros
ruta_params = f"{PARAM['drive_path']}/best_params_global.json"
with open(ruta_params, 'w') as f:
    json.dump(study.best_params, f, indent=2)
print(f'Parámetros guardados en: {ruta_params}')

# guardar trials
ruta_trials = f"{PARAM['drive_path']}/trials_global.csv"
trials_ok.to_csv(ruta_trials, index=False)
print(f'Trials guardados en: {ruta_trials}')

print()
print('Resumen final:')
print(f'  RMSE CV:     {np.mean(rmses_final):.4f} ± {np.std(rmses_final):.4f}')
print(f'  Trials OK:   {len(trials_ok)}')
print(f'  Trials podados: {len(trials_df) - len(trials_ok)}')
print(f'  Features usadas: {len(cols_train)}')

---
# APARTADO — Warm-starting para clusters

**La pregunta**: si separo los productos en clusters y quiero optimizar un LightGBM para cada cluster, ¿tengo que arrancar Optuna desde cero o puedo reutilizar lo que aprendió en la búsqueda global?

**La respuesta**: se puede reutilizar. Optuna guarda cada trial como un objeto con parámetros + resultado. Podés "sembrar" un nuevo estudio con los mejores trials del estudio global — el sampler TPE arranca con esa información y explora desde ahí en lugar de explorar al azar.

**¿Cuándo conviene?**
- Si el cluster tiene pocos productos → poco dato → el modelo es sensible a los hiperparámetros → warm-start ahorra trials
- Si el cluster es muy distinto al promedio global → el warm-start puede confundir → conviene agregar una fase de exploración antes

In [ ]:
def crear_estudio_cluster(study_global, cluster_id, X_cluster, y_cluster,
                          n_trials_warm=20, n_trials_new=40,
                          top_k_trials=10, semilla=102191):
    """
    Crea un estudio Optuna para un cluster específico, sembrado con los
    mejores trials del estudio global.

    Estrategia:
    1. Tomar los top_k_trials mejores del estudio global
    2. Agregarlos al nuevo estudio como trials completados (enqueue)
    3. Correr n_trials_warm trials fijados (evaluar esos parámetros en el cluster)
    4. Correr n_trials_new trials libres de exploración
    """

    # extraer mejores trials del estudio global
    mejores_trials = sorted(
        [t for t in study_global.trials if t.state == optuna.trial.TrialState.COMPLETE],
        key=lambda t: t.value
    )[:top_k_trials]

    # crear nuevo estudio para el cluster
    sampler_cluster = optuna.samplers.TPESampler(seed=semilla + cluster_id)
    study_cluster   = optuna.create_study(
        direction='minimize',
        sampler=sampler_cluster,
        study_name=f'lgbm_cluster_{cluster_id}',
    )

    # sembrar con los mejores parámetros del estudio global
    for trial in mejores_trials:
        study_cluster.enqueue_trial(trial.params)

    def objective_cluster(trial):
        params = {
            'objective':        'regression',
            'metric':           'rmse',
            'verbosity':        -1,
            'n_jobs':           -1,
            'random_state':     semilla,
            'subsample_freq':   1,
            'n_estimators':     trial.suggest_int('n_estimators', 100, 2000),
            'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'num_leaves':       trial.suggest_int('num_leaves', 16, 256),
            'max_depth':        trial.suggest_int('max_depth', 3, 12),
            'min_child_samples':trial.suggest_int('min_child_samples', 5, 100),
            'subsample':        trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'min_split_gain':   trial.suggest_float('min_split_gain', 0.0, 1.0),
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=semilla)
        rmses = []
        for idx_tr, idx_val in kf.split(X_cluster):
            m = lgb.LGBMRegressor(**params)
            m.fit(X_cluster.iloc[idx_tr], y_cluster.iloc[idx_tr],
                  eval_set=[(X_cluster.iloc[idx_val], y_cluster.iloc[idx_val])],
                  callbacks=[lgb.early_stopping(30, verbose=False),
                             lgb.log_evaluation(-1)])
            pred = m.predict(X_cluster.iloc[idx_val])
            rmses.append(mean_squared_error(y_cluster.iloc[idx_val], pred, squared=False))
        return float(np.mean(rmses))

    # fase warm: evaluar los parámetros globales en el cluster
    study_cluster.optimize(objective_cluster,
                           n_trials=min(top_k_trials, n_trials_warm),
                           show_progress_bar=False)

    # fase exploración: búsqueda libre
    study_cluster.optimize(objective_cluster,
                           n_trials=n_trials_new,
                           show_progress_bar=False)

    return study_cluster


print('Función warm-start definida.')
print()
print('Uso:')
print('  study_cl0 = crear_estudio_cluster(study, cluster_id=0, X_cluster=X_cl0, y_cluster=y_cl0)')
print('  print(study_cl0.best_value, study_cl0.best_params)')

In [ ]:
# ── Diagrama conceptual: global → cluster ──
print("""
  ESTUDIO GLOBAL (todos los productos)
  ─────────────────────────────────────
  80 trials → TPE aprende el landscape de hiperparámetros
  best_params: { n_estimators: 800, lr: 0.05, ... }
         │
         │  top_k_trials (ej: 10 mejores)
         │  se copian como seed al nuevo estudio
         ▼
  ESTUDIO CLUSTER 0 (ej: Mayonesas)
  ──────────────────────────────────
  Fase warm  (10 trials): evalúa los parámetros globales en el cluster
  Fase libre (40 trials): TPE explora desde lo que aprendió
  → converge más rápido que arrancar desde cero

  ESTUDIO CLUSTER 1 (ej: Jabones)
  ──────────────────────────────────
  Idem — misma semilla del global, distinto cluster_id

  Ventaja: si Mayonesas y Jabones son similares en comportamiento,
  los parámetros globales ya son un buen punto de partida.
  Si son muy distintos, la fase libre lo corregirá.
""")

In [ ]:
# ── Comparación: ¿mejora el warm-start vs cold-start? ──
# (correr solo si tenés los clusters definidos)

# Ejemplo simulado con el mismo dataset global dividido al azar
# Reemplazar con los clusters reales cuando estén disponibles

RUN_WARMSTART_DEMO = False   # ← poner True para correr la demo

if RUN_WARMSTART_DEMO:
    np.random.seed(42)
    mask = np.random.rand(len(X)) < 0.5
    X_demo = X[mask]; y_demo = y[mask]

    # cold start
    study_cold = optuna.create_study(direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=0))
    study_cold.optimize(
        lambda t: objective(t),   # noqa — usa X y y globales solo para demo
        n_trials=30, show_progress_bar=True)

    # warm start
    study_warm = crear_estudio_cluster(study, cluster_id=99,
        X_cluster=X_demo, y_cluster=y_demo,
        n_trials_warm=10, n_trials_new=20)

    # comparar curvas de convergencia
    def best_so_far(s):
        vals = [t.value for t in s.trials if t.state == optuna.trial.TrialState.COMPLETE]
        return np.minimum.accumulate(vals)

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=best_so_far(study_cold), name='cold start', line=dict(color='tomato')))
    fig.add_trace(go.Scatter(y=best_so_far(study_warm), name='warm start', line=dict(color='steelblue')))
    fig.update_layout(title='Cold start vs Warm start — convergencia RMSE',
                      xaxis_title='trial', yaxis_title='RMSE', height=400)
    fig.show()